# 한국철도공사 역별 주차장 현황 분석 (2023-08-01 기준)

한국철도공사가 공개한 '역사별 주차장 현황' 데이터를 정제해, 철도역 주차 인프라의 **공급 규모와 지역별 격차**를 분석합니다.

- 데이터: 한국철도공사_역사별 주차장 현황_20230801.csv (207개 주차장, 187개 역)
- 컬럼: 개소수, 주차장명, 관할 지역본부, 역, 주차대수

**중요 — 이 데이터로 할 수 있는 것 / 없는 것**

이 데이터는 2023-08-01 기준 주차 **"대수"(설치 규모) 스냅샷 한 장**입니다. 실제 이용 건수, 시간대별/요일별 수요, 시계열 변화가 없어 **이용률·포화도·피크 수요 예측·투자 회수 기간** 같은 분석은 이 데이터만으로 할 수 없습니다. 대신 이 노트북은 **공급(규모) 격차** — 어느 역·지역본부가 상대적으로 크거나 작은 주차장을 갖고 있는지 — 를 정량적으로 분석하고, 이용 데이터 확보가 왜 다음 단계로 필요한지를 액션 플랜에서 짚습니다.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('../..').resolve()))
from common.data_utils import read_csv_any_encoding

plt.rcParams['axes.unicode_minus'] = False
# plt.rc('font', family='NanumGothic')  # 한글 폰트가 있다면 주석 해제 (Linux/Windows)
# plt.rc('font', family='AppleGothic')  # macOS

RAW = Path('raw/한국철도공사_역사별 주차장 현황_20230801.csv')

## 1. 데이터 로드 및 정제

In [ ]:
df = read_csv_any_encoding(RAW)
print(df.shape)
df.head()

### 결측·이상치 확인

결측치가 2건 있습니다.

In [ ]:
print(df.isnull().sum())
df[df['역'].isnull() | df['주차대수'].isnull()]

**발견한 원자료 이슈**: `광명B` 행은 `역`이 비어 있고 `관할 지역본부`가 지역명이 아니라 위탁 운영사명(`HS홀딩스(위탁)`)으로 되어 있습니다. `주차장명`이 `광명A`(수도권광역본부, 광명역)와 짝을 이루는 것으로 보아, 위탁 운영으로 넘어가면서 역/지역본부 정보가 누락된 것으로 판단됩니다 → **광명역의 두 번째 주차장**으로 보정합니다.

`묵호역`은 `주차대수`가 결측이라 규모 집계에서는 제외합니다(주차장 존재 자체는 유지).

In [ ]:
fix_mask = df['주차장명'] == '광명B'
df.loc[fix_mask, '역'] = '광명역'
df.loc[fix_mask, '관할 지역본부'] = '수도권광역본부'  # 자매 주차장 '광명A'와 동일 지역본부로 보정

print(f"고유 역 수: {df['역'].nunique()}개 / 전체 주차장 수: {len(df)}개")
print(f"주차대수 결측(집계 제외): {df['주차대수'].isnull().sum()}건")

## 2. 역별 규모 분석

한 역에 여러 주차장이 있는 경우가 많아(예: 광명역 A/B), 역 단위로 합산해서 봅니다.

In [ ]:
by_station = (
    df.groupby('역')
    .agg(주차장수=('주차장명', 'count'), 총주차대수=('주차대수', 'sum'))
    .sort_values('총주차대수', ascending=False)
)
top15 = by_station.head(15)

fig, ax = plt.subplots(figsize=(8, 6))
top15.iloc[::-1]['총주차대수'].plot.barh(ax=ax, color='#12897B')
ax.set_title('역별 총 주차대수 Top 15')
ax.set_xlabel('주차대수')
plt.tight_layout()
plt.show()

top15

**인사이트**: 광명역이 두 주차장을 합쳐 2,791대로 압도적 1위입니다(KTX 환승 대형 주차타워로 잘 알려진 곳과 일치). 광주송정역(1,886대), 천안아산역(1,047대)이 뒤를 잇습니다. 상위 10개 역이 전체 주차대수의 39.2%를 차지해, 대형 환승역 중심으로 공급이 집중되어 있습니다.

## 3. 지역본부별 공급 격차

In [ ]:
region = (
    df.groupby('관할 지역본부')
    .agg(주차장수=('주차장명', 'count'), 총주차대수=('주차대수', 'sum'), 역수=('역', 'nunique'))
)
region['역당평균'] = (region['총주차대수'] / region['역수']).round(1)
region = region.sort_values('역당평균', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
region['역당평균'].plot.bar(ax=ax, color='#12897B')
ax.set_title('지역본부별 역당 평균 주차대수')
ax.set_ylabel('역당 평균 주차대수')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

region

In [ ]:
capital = ['서울본부', '수도권광역본부']
cap = df[df['관할 지역본부'].isin(capital)]
noncap = df[~df['관할 지역본부'].isin(capital)]

cap_avg = cap['주차대수'].sum() / cap['역'].nunique()
noncap_avg = noncap['주차대수'].sum() / noncap['역'].nunique()

print(f"수도권(서울본부+수도권광역본부): {cap['역'].nunique()}개 역, 역당 평균 {cap_avg:.1f}대")
print(f"비수도권: {noncap['역'].nunique()}개 역, 역당 평균 {noncap_avg:.1f}대")
print(f"비수도권이 수도권보다 역당 평균 {(noncap_avg/cap_avg - 1)*100:.0f}% 더 큼")

**인사이트 (반직관적)**: 서울본부는 역 수(39개)·주차장 수(47개)가 가장 많지만, **역당 평균 주차대수는 92.7대로 전 지역본부 중 가장 작습니다**. 수도권 전체로 봐도 역당 평균(128.1대)이 비수도권(180.5대)보다 오히려 **41% 작습니다**.

수도권은 인프라가 가장 잘 갖춰져 있을 것이라는 통념과 반대로, 서울/수도권 역은 대중교통 접근성이 높아 주차 수요가 상대적으로 분산되고 부지 확보도 어려운 반면, 비수도권 역은 자가용 의존도가 높아 역 하나당 더 큰 환승주차장을 운영하는 것으로 해석할 수 있습니다. (단, 이는 공급 규모 기준 해석이며 실제 이용률 데이터로 검증이 필요합니다 — 4절 참고)

## 4. 개선 액션 플랜

### 즉시 실행 (0-3개월)
1. **이용 데이터 연계 확보** — 이 데이터는 공급 규모(설치 대수)만 담고 있어 포화도·피크 수요를 알 수 없습니다. 코레일 주차장 입출차 시스템의 실제 이용 데이터를 연계해야 "부족/과잉" 판단이 가능합니다. 이후 모든 액션의 전제 조건입니다.
2. **광명역급 대형 거점 우선 실태조사** — 상위 10개 역(전체 공급의 39%)부터 현장 실태조사를 진행해 데이터 공백을 빠르게 메웁니다.

### 중기 과제 (3-12개월)
1. **비수도권 대형 주차장의 실사용률 점검** — 비수도권 역당 평균이 수도권보다 41% 큰 것이, 실제 높은 이용률 때문인지 과잉 공급인지 이용 데이터로 검증합니다.
2. **서울본부 역세권 대체 접근성 확대** — 역당 평균이 가장 작은 서울본부는 물리적 확장이 어려우므로, 공유 모빌리티·환승 셔틀 등 주차 대체 수단을 우선 검토합니다.

### 장기 과제 (1년 이상)
1. **지역본부 간 공급 형평성 기준 수립** — 역당 평균 격차(72~308대, 지역본부 간 4배 이상 차이)를 설명 가능한 기준(승하차 인원, 배후 인구 등) 대비로 재평가해 신증설 우선순위에 반영합니다.

## 5. 데이터 한계 및 방법론 노트

- 이 데이터는 **2023-08-01 단일 시점 스냅샷**이며 시계열이 없어 추세·계절성 분석이 불가능합니다.
- 실제 이용 건수·회전율·포화도 데이터가 없어, 앞서 언급한 "이용률/피크 수요/투자 회수 기간" 관련 분석은 이 데이터만으로 수행할 수 없습니다.
- `광명B` 행은 `역`·`관할 지역본부`가 원자료에 누락/오기되어 있어(위탁 운영사명이 지역본부 자리에 들어가 있음), 자매 주차장 `광명A` 기준으로 보정했습니다.
- `묵호역` 1건은 `주차대수`가 결측이라 규모 집계에서 제외했습니다(전체 187개 역 중 1개 역만 영향).
- 본 분석은 서술 통계 기반이며, 제시된 관계는 상관관계로 인과관계를 증명하지 않습니다.